# Generalised Chemical Engine
In previous iterations of this project, specific chemical systems (such as the Haber-Bosch process) were modelled using hard-coded differential equations. While effective for isolated cases, that approach lacks scalability. If we wish to investigate a new reaction mechanism, we must rewrite the underlying mathematics.

This section introduces a **generalised chemical solver**. Instead of writing new code for every reaction, we build a "summation machine" that can simulate *any* arbitrary mechanism defined by the user. This allows us to investigate complex, multi-step phenomena, such as the oxidation of nitric oxide ($2NO + O_2 \rightarrow 2NO_2$), which involves transient intermediates and timescale discrepancies ("stiffness").

The system parses a network of chemical species and elementary steps, constructs the necessary differential equations dynamically, and solves them using advanced numerical integration techniques.

## 1. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp
from IPython.display import display
import ipywidgets as widgets
import pandas as pd
import copy
from typing import List, Tuple, Optional, Dict, Set

## 2. Model Configuration
This defines the physical and chemical constants of the simulation.

In [2]:
# physical constants...
R_J_MOL_K = 8.314 # universal gas constant in J K^-1 mol^-1
R_ATM_L = 0.08206 # universal gas constant in L atm K^-1 mol^-1, for pressure calculations

## 3. The Object-Oriented Engine
To move beyond hard-coded models (which are limited to specific reactions), we introduce a generalised architecture. This allows us to assemble *any* reaction mechanism from its fundamental components: species and elementary steps.

### 3.1 The Chemical Species
The `ChemicalSpecies` class acts as the digital identity for a molecule. Instead of looking up physical constants in global dictionaries, each species object carries its own intrinsic properties.
* **Identity:** name (e.g., "$NO$", "$N_2O_2$").
* **Real gas properties:** encapsulates the Van der Waals constants ($a$ and $b$), allowing the simulation to automatically calculate real gas pressures for *any* mixture without hard-coded mixing rules.

To accurately model complex chemical systems, it is often necessary to represent a species whose concentration is held constant. This could be a buffered species, a solvent, or a reactant supplied by a continuous external source.

To handle this, the `ChemicalSpecies` class accepts a `species_type` argument. 
* `species_type='reactant'` (Default): the species' concentration evolves according to reaction stoichiometry.
* `species_type='pool'`: the species' concentration is treated as a fixed parameter throughout the simulation. It contributes to reaction rates but is not consumed or produced. The influence of pool chemicals is calculated dynamically during the simulation.

The internal ODE solver constructs a reduced state vector containing only the `reactant` species. This improves performance by not solving for derivatives that are known to be zero. Moreover, there is an automatic validation check ensuring that no reaction attempts to produce a pool chemical.

### 3.2 The Elementary Reaction
The `Reaction` class models a single, unidirectional elementary step. By treating reversible reactions as two distinct steps (forward and reverse), the mathematical logic is simplified: every reaction is just a consumption of reactants to form products.

**The Rate Law:**

The class implements the general Law of Mass Action. For a reaction $aA + bB \rightarrow P$, the rate $v$ is calculated as:
$$ v(T) = k(T) \cdot [A]^a [B]^b $$

**Temperature Dependence:**

The rate constant $k(T)$ is determined dynamically using the Arrhenius Equation, ensuring the simulation responds correctly to temperature changes:
$$ k(T) = A \cdot e^{-\frac{E_a}{RT}} $$

This architecture allows us to construct complex, multi-step mechanisms (like the oxidation of nitric oxide) simply by instantiating a list of `Reaction` objects.

**Dynamic Equilibrium Detection**

A key design goal of this project is to create a truly *generalised* engine. This meant that thermodynamic analysis (`Qc`, `Kc`, etc.) could not be hard-coded. Previous versions simply assumed that the primary equilibrium was defined as the first and second reactions in the system. This was a fragile assumption that forced the user to arrange their reactions in a specific, non-obvious way and would fail on more complex mechanisms.

To solve this, the `find_all_equilibrium_pairs` function was implemented. This utility function inspects the entire list of reactions provided by the user:
1. It iterates through every possible unique pair of reactions in the `reaction_list`.
2. For each pair, it performs a logical check: *are the reactants of reaction A identical to the products of reaction B, AND are the products of reaction A identical to the reactants of reaction B?* (if so, this is an equilibrium)
3. It finds **all** such pairs, ensuring that once a reaction is part of a detected pair, it is not considered again.

Through this upgrade, the engine is now "smarter" and more intuitive. It automatically handles the thermodynamic analysis for any valid mechanism containing a reversible step, without requiring special configuration from the user. It also allows other parts of the system to identify chemically relevant equilibria (such as a steady state) without needing hard-coded indices.

### 3.3 The Generalised System

The `ChemicalSystem` class serves as the universe in which the simulation runs.

**The Mathematics of Coupled Reactions:**

In a multi-step mechanism, a single species often participates in multiple reactions simultaneously. For example, in the oxidation of nitric oxide:
1.  $2NO \rightarrow N_2O_2$ (consumes $NO$ and produces $N_2O_2$)
2.  $N_2O_2 \rightarrow 2NO$ (produces $NO$ and consumes $N_2O_2$)
3.  $N_2O_2 + O_2 \rightarrow 2NO_2$ (consumes $N_2O_2$ and $O_2$)

How does the solver handle $N_2O_2$, which is being formed in step 1, consumed in step 2, and consumed in step 3?

The `get_net_rates` method solves this by implementing the following: the net rate of change for any species $i$ is simply the sum of its stoichiometric contribution from every reaction $j$ in the network:

$$ \frac{dn_i}{dt} = V \sum_{j} \nu_{ij} r_j $$

Where:
-   $V$ is the system volume.
-   $r_j$ is the rate of reaction $j$.
-   $\nu_{ij}$ is the stoichiometric coefficient of species $i$ in reaction $j$ (negative for reactants, positive for products).

A previous version calculated derivatives by looping through lists of objects. To optimise performance for stiff solvers (which evaluate derivatives thousands of times), we convert the reaction mechanism into a **stoichiometry matrix**, denoted as $\mathbf{S}$.

$$ \mathbf{S} = \begin{bmatrix}
\nu_{1,1} & \nu_{1,2} & \dots \\
\nu_{2,1} & \nu_{2,2} & \dots \\
\vdots & \vdots & \ddots
\end{bmatrix} $$

Where $\nu_{i,j}$ represents the stoichiometric coefficient of species $i$ in reaction $j$.

*Dimensions:*
*   Rows ($N$): represent the chemical species.
*   Columns ($M$): represent the elementary reactions.

The net rate of change for all species vector $\frac{d\mathbf{n}}{dt}$ can then be calculated in a single CPU operation using the dot product of the matrix $\mathbf{S}$ and the reaction rate vector $\mathbf{r}$:

$$ \frac{d\mathbf{n}}{dt} = V \cdot (\mathbf{S} \cdot \mathbf{r}) $$

This operation condenses the entire differential equation system and massively accelerates the solver.

**Stiffness**

The mechanism for NO oxidation introduces a computational challenge known as **stiffness**.
1.  **Fast Step:** $2NO \rightleftharpoons N_2O_2$ establishes equilibrium almost instantly ($k \approx 10^{13}$).
2.  **Slow Step:** $N_2O_2 + O_2 \to 2NO_2$ proceeds much slower ($k \approx 10^9$).

Standard explicit integrators (like Runge-Kutta 45) fail here. If they use a time-step small enough to capture the fast equilibrium, the simulation takes forever. If they use a large time-step, the fast reaction becomes numerically unstable.
*   **Solution:** we utilise the **Radau** method. These are *implicit* integration methods. These can take large time steps where explicit methods would fail, although computational overhead per step is higher.

The `run_simulation` method sets `dense_output=True` when calling the solver. 
*   **The problem:** implicit solvers (like Radau) take huge time steps. A 10-second simulation might only have 5 actual calculated steps. Plotting these straight lines would look jagged and miss the curvature of the reaction.
*   **The solution:** `dense_output` constructs a continuous polynomial interpolant of the solution. This allows us to evaluate the reaction state at *any* point in time, not just the steps the solver took, resulting in perfectly smooth curves for visualisation without the computational cost of taking tiny steps.

**Configuring the Numerical Solver**

Although Radau is the default (as it is accurate, general-purpose, and excellent for most problems), when a `ChemicalSystem` object is created, there is an option to choose which solver to use and set the solver tolerances. For example, BDF (a specialised solver that is often faster and more stable for extremely stiff systems) might be preferred.

```python
system = ChemicalSystem(
    species_list, 
    reaction_list, 
    initial_moles, 
    initial_V=1.0, 
    initial_T=300,
    method='BDF',      # specify the BDF solver
    rtol=1e-4,         # relax the relative tolerance for speed
    atol=1e-7)
```

**The Iterative Chunking Solver**

Real chemical systems do not have a pre-defined "end time." A reaction might take 1 second or 100 years depending on the temperature. To handle this, the `run_simulation` method employs the following approach:
1.  **Initial guess:** the simulation attempts to solve for a short duration (e.g., 0.1s).
2.  **Reaction completion check:** it analyses the derivatives (net rates) at the end of the chunk. If the maximum rate of change is below a strict tolerance (e.g., $10^{-7}$ mol/s), the reaction(s) is deemed completed.
3.  **Expansion:** if not completed, the simulation extends the time horizon geometrically (multiplying the duration by 10) and continues from the last state.

This ensures the simulation runs for exactly as long as necessary.

A new internal method, `_get_characteristic_timscales`, first analyses the entire reaction mechanism at the initial temperature. It estimates a characteristic time for each step based on the rate constant and the initial rate of consumption of each reactant. It iterates through every reaction and generates a logarithmically spaced time grid that clusters points specifically around these characteristic times. This guarantees the resolution of the simulation automatically scales to match the physics of the reaction, capturing femtosecond dynamics and hour-long decays in the same dataset.

The `run_simulation` method uses these detected timescales to construct a composite array of time points for the solver. It generates a high density of points around *each* characteristic time.

This ensures that we capture high-resolution data during every kinetically significant event, whether it's a fast equilibrium at the start or a rapid decomposition step occurring much later in the simulation.

**Fixed-Duration Simulation Mode**

Previously, the engine operated exclusively in the convergence mode, where it ran until the net reaction rates dropped below a threshold.

While that is ideal for thermodynamic systems, it fails for other systems, such as:
*   *Oscillating reactions:* systems like Lotka-Volterra never reach a standstill; they could cycle indefinitely. Trying to wait for "equilibrium" causes the engine to run unnecessarily until it hits the `max_iterations` limit.
*   *Stiff kinetics:* when analysing extremely fast initial transients, we often want to simulate just the first few milliseconds without forcing the solver to compute the subsequent hours of slow decay.

The solution was to add an optional `t_end` parameter:
*   If `t_end` is None (default): the engine runs the standard iterative chunking solver.
*   If `t_end` is set: the engine simply integrates the differential equations from $t=0$ to $t=t_{end}$.

**Other Features**

SciPy's solvers expect a generic function signature `f(t, y)`. However, our chemical engine operates on dictionaries and objects.
*   This method acts as a translator, converting the raw NumPy array from the solver (`y`) into a semantic dictionary (`{'NO': 0.1, ...}`), passes it to the chemistry engine to calculate rates, and converts the result back to an array. 

It also applies a safety clamp which prevents numerical noise from creating negative mass, which could cause complex number errors when calculating rates with fractional orders (e.g., $\sqrt{\text{concentration}}$).

#### Thermodynamics
This engine adheres to the strict kinetic definition:
$$K_c(T) = \frac{k_{forward}(T)}{k_{reverse}(T)}$$
By deriving $K_c$ directly from the Arrhenius parameters of the forward and reverse steps, the engine guarantees that the thermodynamic limit is mathematically consistent with the reaction kinetics.

At every time step, the engine calculates the reaction quotient ($Q$) which measures the relative amounts of products and reactants at that specific moment.
*   **$Q < K$:** the system will drive forward.
*   **$Q > K$:** the system will drive backward.
*   **$Q = K$:** the system is at equilibrium.

The engine calculates two distinct versions of these metrics:
*   **Concentration ($Q_c$, $K_c$):** based on molarity ($[X]=n/V$). This is calculated for all species, regardless of phase.
*   **Pressure ($Q_p$, $K_p$):** based on partial pressures. 

#### Real Gases
At high pressures or low temperatures, the behaviour of gases deviate from that described by the ideal gas law ($PV=nRT$). The engine implements the Van der Waals equation to model real gases:
$$(P+\frac{an^2}{V^2}(V-nb)=nRT)$$
By storing the Van der Waals constants ($a$ and $b$) in each `ChemicalSpecies` object, the engine calculates real pressure and the compressibility factor, ensuring an accurate physical model.

**Real Gas Mixing Rules**

Applying the Van der Waals equation to a dynamic mixture of evolving species requires complex mixing rules. In `calculate_pressure_array`, the engine calculates the effective coefficients $a_{mix}$ and $b_{mix}$ at every time step based on the changing mole fractions ($x_i$).

1. The attraction parameter ($a_{mix}$): we use a quadratic mixing rule...
$$\sqrt{a_{mix}} = \sum_{i} x_i \sqrt{a_i} \implies a_{mix} = \left( \sum_{i} x_i \sqrt{a_i} \right)^2$$
2. The volume parameter ($b_{mix}$): we use a linear combination...
$$b_{mix} = \sum_{i} x_i b_i$$

The simulation then solves the cubic Van der Waals equation for pressure:
$$P_{real} = \frac{n_{total}RT}{V - n_{total}b_{mix}} - \frac{a_{mix}n_{total}^2}{V^2}$$

### Phases
The engine recognises 4 states of matter: `gas`, `liquid`, `solid`, and `aqueous`. This is defined during `ChemicalSpecies` initialisation.

System pressure ($P_{real}$ and $P_{idea}$) is now derived exclusively from moles of species marked as `gas`. Van der Waals mixing rules are applied only to the gaseous components of the mixture.

### Yield Calculation
The `calculate_yield(product, limiting reactant, ratio)` calculates the percentage yield of a specific product. The theoretical yield is derived from the initial moles of the `limiting_reactant` multiplied by the stoichiometric `ratio` ($S=\frac{\text{coefficient of product}}{\text{coefficient of reactant}}$), and the actual yield is taken from the final simulation state.

This is a post-processing tool that does not affect the simulation itself.

### Solvents
A `ChemicalSpecies` can be designated as a solvent by setting `phase='solvent'`.
* **Auto-initialisation:** if a solvent is defined with `0.0` initial moles, the engine automatically calculates the moles required to fill the system volume $V$ based on the species' `density` and `molar_mass`.
$$n_{solvent}=V_{sys}\times\frac{\rho_{solvent}}{MM_{solvent}}$$
(*default parameters for water: $\rho=1000$ g/L, molar mass $=18.015$ g/mol*)

In rate equations, the solvent is treated as a standard reactant. Its concentration (for water: $n/V\approx55.5 M$) **is** included in the rate calculation. Rate constants ($A$) must account for this. 

In calculations of $Q_c$ and $K_c$, the solvent is assigned a thermodynamic activity of 1.0 (unity). This ensures that the presence of the solvent does not skew the equilibrium position calculation, aligning with the standard convention where solvent is omitted.

### 3.4 Perturbations
To simulate Le Chatelier's Principle, we need to stress the system. Simulating a perturbation (e.g., a sudden volume expansion) creates a mathematical discontinuity in the differential equations. Standard solvers cannot integrate across discontinuities. The `PerturbationSimulation` class handles this via a 3-stage algorithm::
1.  **History:** extracts the pre-perturbation state. The baseline is sliced up to $t_{start}$.
2.  **Stress:** runs a simulation where volume, temperature, or moles change over time. A dedicated solver runs from $t_{start}$ to $t_{end}$. During this phase, the `_ode_adapter_perturbation` method intercepts the derivatives and models a perturbation.
    *   *Volume/Temperature:* uses linear interpolation ($V(t)$) to modify conditions dynamically.
        * This allows the engine to model temperatures changes, pressure changes, volume expansions/compressions, and more.
    *   *Injection:* adds an external flow rate term to the differential equations: $dn/dt_{total} = dn/dt_{rxn} + rate_{inj}$.
        * This allows the user to model scenarios such as the slow addition of a reagent to a titration or the continuous fuel injection in a combustion engine, or removal of a desired product. 
3.  **Relaxation:** simulates the return to equilibrium after the stress ends. The final state vector $y$ of stage 2 is used as the initial condition $y_0$ for a new simulation instance, representing the system's return to equilibrium.

**Stitching**

Finally, the arrays are concatenated. Crucially, the engine performs deduplication: because the end of stage 1 and start of stage 2 are mathematically identical time points, floating-point noise can cause $dt\approx0$ errors in rate calculations. The `sanitise_results` function filters these artifacts using a mask `np.diff(t) > 1e-9` to ensure numerical safety for downstream analysis.

**Sequential Perturbations**

The engine supports multi-stage industrial processes (like the repeated extraction of ammonia in the Haber process). Since the `run_simulation` and `PerturbationSimulation` methods output the precise `final_moles` and `final_V`, these can be fed back into a new `ChemicalSystem` instance as the `initial_moles`.

## 4. Visualisation and Data Handling
### 4.1 Data Sanitisation (`sanitise_results`)
The adaptive solver (Radau/LSODA) is excellent at physics but messy with data. It often outputs duplicate time points or extremely small negative values due to floating-point noise.

This function cleans the raw output:
1.  Removes points where $dt \approx 0$ to prevent "divide by zero" errors when calculating rates ($\frac{d[C]}{dt}$).
2.  Ensures that derived datasets (like $Q_c$ arrays) are filtered in sync with the main time array.

**Automatic Graph Trimming**

`find_completion_time` is an analysis function which detects when the reaction is complete, automatically trimming the graph to the most relevant time window. It determines completion by satisfying two distinct criteria simultaneously:
* It monitors the rate of change of all species. The system is considered stable only when the fastest-changing species has a rate below a strict numerical tolerance (e.g., $10^{-7}$ mol/s). This confirms that the net reaction has ceased.
* For any reversible reaction pairs in the mechanism, it checks that the forward rate has become approximately equal to the reverse rate ($Rate_{fwd} \approx Rate_{rev}$). This confirms that dynamic equilibria or steady states have been fully established.

The graph's x-axis is then automatically set to end shortly after the *later* of these two conditions is met. This provides a clean, focused view of the entire reaction arc without displaying unnecessary information.

Another function, `align_previous_run`, was created to align data from a previous run with the current run, allowing for plotting comparisons.

### 4.2 The Dashboard (`generate_table` and `generate_plot`)
These two functions produces the graphs and tables, along with doing the following:
*   **Extrapolation:** if comparing a new run to a shorter previous run, it mathematically extends the previous data (holding values constant) to ensure the x-axes align perfectly.
*   **Log-scale visualisation:** since intermediates like $N_2O_2$ exist at concentrations $10^4$ to $10^6$ times lower than reactants, linear plots render them invisible. The option to have logarithmic scales for time and concentrations is offered.
*   **Thermodynamic:** tracks the reaction quotient ($Q_c$) relative to the equilibrium constant ($K_c$), along with $Q_p$ and $K_p$. In coupled mechanisms, a permanent gap between these lines indicates a **steady state** rather than a true equilibrium.
*   **Dual-Axis Visualization:**
    *   *Top:* concentrations (log scale) and real pressure (teal, linear).
    *   *Bottom:* rates and thermodynamics ($Q_c/K_c$, purple).
*   **View window:** the user can specify a zoom window if they would like to focus on a particular event/area.
*   **Perturbation zoom:** a separate graph is produced during perturbations showing detail around when the perturbation occurred, giving more details.

### 4.3 Timeline
Chemical reactions occur on logarithmic timescales. For example, with the oxidation of nitric oxide, the initial equilibrium ($2NO \rightleftharpoons N_2O_2$) is established in microseconds ($10^{-6} s$), while the bulk oxidation takes minutes ($100 s$).

To solve this, a new analysis method, `find_key_events`, has been introduced. After a simulation completes, this function analyses the results to find the precise timestamps of critical physical events:
*   **Equilibrium convergence:** when a reversible pair's forward and reverse rates become equal.
*   **Intermediate peaks:** when a transient species reaches its maximum concentration.
*   **Reactant half-life:** when a starting material is 50% consumed.

The user interface then uses this information to **dynamically generate a set of "Focus" buttons.** Each button is labeled with the event and its time. Clicking a button automatically zooms the graph to that specific moment, allowing the user to instantly investigate the most scientifically interesting parts of the reaction timeline without manual searching. This allows for more interesting analysis.

### 4.4 User Interface
Unlike the previous Haber Process interface, which had fixed sliders, this function generates the UI **dynamically** by inspecting the `ChemicalSystem` object.
1.  **Species sliders:** it iterates through `system.species_list` to create a slider for every molecule in the simulation.
2.  **View control:** adds a "Graph View Window" slider, allowing the user to zoom into the fast initial kinetics or scroll to the steady state without re-running the solver.
3.  **Perturbations:** links the simulation engine to the perturbation engine, creating UI to handle volume, temperature, and injection events.

This interface uses a GUI state object (`GUIState`) to manage application state across user interactions. Because `ipywidgets` is event-driven and callbacks do not share local variables, `GUIState` acts as a central memory for the GUI, storing the current and previous simulation systems, active perturbation windows, and selected thermodynamic analysis options. This separation keeps the user interface logic decoupled from the chemical simulation logic and enables features such as before/after comparisons, perturbation highlighting, etc.

#### Perturbation Time Window
The linear time slider has been replaced by a scientific notation input system. This addresses the stiffness of chemical simulations, where critical events could occur across vastly different timescales.

The perturbation window $[t_{start}, t_{end}]$ is now constructed via four different controls:
1. Mantissa ($M$): a float slider ($0.00-9.99$).
2. Exponent ($E$): an integer slider ($-15$ to $+10$).
$t$ is calculated as:
$$t=M\times10^E$$
This separates the precision of the selection from the magnitude of the time.

### 4.5 The Steady State Approximation

In the generated plots of certain reactions, a persistent gap between the reaction quotient ($Q_c$, purple dashed) and the equilibrium constant ($K_c$, violet dot-dash) may be observed. In the simple reactions such as the Haber Process, these lines always converged. When running something like the oxidation of nitric oxide, they run parallel but distinct. **Why?**

This is the visual signature of a **pseudo-steady state**.

1.  Reaction 1 ($2NO \to N_2O_2$) tries to fill the "bucket" of $N_2O_2$ up to its equilibrium level ($K_c$).
2.  Reaction 2 ($N_2O_2 + O_2 \to 2NO_2$) constantly drains the bucket to form products.

Because the intermediate is being consumed as fast as it is formed, it never quite reaches the full concentration that thermodynamics ($K_c$) predicts. It reaches a dynamic balancing point where:
$$ \text{rate of formation} \approx \text{rate of consumption} $$
$$ k_1[NO]^2 \approx k_{-1}[N_2O_2] + k_2[N_2O_2][O_2] $$

The "distance" on the graph represents the term $k_2[O_2]$. If reaction 2 were slower, the gap would close. If reaction 2 were faster, the gap would widen.

In multistep reaction mechanisms, we often encounter highly reactive intermediates - species that are produced and then consumed almost immediately. In the example case of oxidation of nitric oxide, the dimer **dinitrogen dioxide ($N_2O_2$)** behaves exactly this way.

**The Concept**

Because the intermediate consumes itself (by falling back to reactants or reacting forward) as fast as it is created, its concentration remains small and relatively constant compared to the major reactants.
Mathematically, we approximate:
$$ \frac{d[N_2O_2]}{dt} \approx 0 $$
This is the steady state approximation.

### Verifying the Gap...
We observe that the reaction quotient ($Q_c$) for the first step ($2NO \rightleftharpoons N_2O_2$) never reaches the equilibrium ($K_c$). We can verify this mathematically.

**1. The Rates**
*   **Step 1 (forward):** formation of $N_2O_2$ from $NO$.
    $$ R_{1f} = k_1 [NO]^2 $$
*   **Step 1 (reverse):** decomposition of $N_2O_2$ back to $NO$.
    $$ R_{1r} = k_{-1} [N_2O_2] $$
*   **Step 2:** consumption of $N_2O_2$ to form $NO_2$.
    $$ R_{2} = k_2 [N_2O_2] [O_2] $$

**2. The Steady State Assumption**

We assume the concentration of the intermediate $[N_2O_2]$ remains relatively constant compared to the major reactants (it is consumed as fast as it forms).
$$ \frac{d[N_2O_2]}{dt} \approx 0 $$
$$ \text{rate of formation} = \text{total rate of consumption} $$
$$ R_{1f} = R_{1r} + R_{2} $$

**3. Solving for Steady State Concentration**

Substitute the rate laws into the balance equation:
$$ k_1 [NO]^2 = k_{-1} [N_2O_2]_{ss} + k_2 [N_2O_2]_{ss} [O_2] $$
Factor out $[N_2O_2]_{ss}$:
$$ k_1 [NO]^2 = [N_2O_2]_{ss} (k_{-1} + k_2 [O_2]) $$
Rearrange to solve for the concentration:
$$ [N_2O_2]_{ss} = \frac{k_1 [NO]^2}{k_{-1} + k_2 [O_2]} $$

**4. Comparing to Equilibrium ($K_c$)**

If Step 2 did not exist ($k_2 = 0$), the system would reach true equilibrium:
$$ [N_2O_2]_{eq} = \frac{k_1 [NO]^2}{k_{-1}} = K_c [NO]^2 $$

**5. The Ratio ($Q_c / K_c$)**

The graph plots $Q_c = \frac{[N_2O_2]}{[NO]^2}$. Let's compare the steady state $Q$ ($Q_{ss}$) to the squilibrium constant $K_c$.

$$ \frac{Q_{ss}}{K_c} = \frac{\left( \frac{[N_2O_2]_{ss}}{[NO]^2} \right)}{\left( \frac{k_1}{k_{-1}} \right)} $$

Substitute the expression for $[N_2O_2]_{ss}$ from Step 3:
$$ \frac{Q_{ss}}{K_c} = \frac{ \left( \frac{k_1}{k_{-1} + k_2 [O_2]} \right) }{ \left( \frac{k_1}{k_{-1}} \right) } $$

Simplify the fraction:
$$ \frac{Q_{ss}}{K_c} = \frac{k_{-1}}{k_{-1} + k_2 [O_2]} $$

The ratio $\frac{Q_c}{K_c}$ represents the fraction of the intermediate that survives to reverse back to reactants versus the total destruction of the intermediate. Since $k_2[O_2] > 0$, this ratio is always $< 1$, hence why the gap exists.

**Verifying the Gap**

Building upon the automated equilibrium detection, the `diagnose_and_verify_steady_state` function was created to verify the steady state. 
1.  *Decomposition:* the function begins by calling `find_all_equilibrium_pairs` to deconstruct the mechanism into its fundamental components: reversible equilibrium pairs and irreversible "drain" reactions.
2.  *Scenario identification:* it then programmatically searches for a "link" between these components. It identifies a steady-state scenario whenever a product from an equilibrium (an intermediate) is consumed as a reactant in one of the drain reactions. It can detect multiple such scenarios in a complex network.
3.  *Mathematical verification:* For each scenario it identifies, it applies the steady-state assumption formula to calculate the *theoretical* `Qc/Kc` ratio that should exist at the end of the simulation:
    $$ \frac{Q_{ss}}{K_c} = \frac{k_{reverse}}{k_{reverse} + \text{drain term}} $$
    The "drain term" is calculated dynamically based on the specific drain reaction's rate law.
4. *Reporting:* finally, it compares this theoretical value to the actual `Qc/Kc` ratio from the simulation results and confirms whether the "gap on the graph" is physically correct or a numerical artifact.

### 4.6 Real Gas Divergence
In standard chemistry texts, we often derive $K_p$ from $K_c$ using the Ideal Gas Law ($PV = nRT$):
$$ K_p \approx K_c (RT)^{\Delta n} $$

However, this simulation calculates pressure using the **Van der Waals** equation of state, which accounts for molecular size ($b$) and intermolecular attraction ($a$). At high pressures or low temperatures, the "real pressure" ($P_{real}$) deviates significantly from the "ideal pressure" ($P_{ideal}$).

**The Divergence**

The diagnostic function `diagnose_real_gas_divergence` quantifies this deviation.
1.  **Ideal $K_p$:** calculated as $K_c (RT)^{\Delta n}$. This assumes molecules are point masses with no forces.
2.  **Observed $K_p$ (real):** this is calculated using the actual $P_{real}$ of the system.

**A Nuance**

When analysing the gap between these values, we must distinguish between two causes:
1.  If the reaction reaches true equilibrium, $Q_p$ will stabilize. The remaining gap between $Q_p$ and ideal $K_p$ is due purely to non-ideal gas behavior (compressibility $Z \neq 1$).
2.  If the reaction involves an intermediate in a pseudo-steady state (like $N_2O_2$), $Q_p$ will stabilise *below* the true equilibrium constant because the intermediate is being constantly drained.

**Interpretation:**

*   If testing a simple equilibrium ($A \rightleftharpoons B$): the gap is purely real gas physics.
*   If testing a coupled mechanism ($2NO \to N_2O_2 \to \dots$): the gap is a sum of different things.

**Compressibility Factor ($Z$)**

This simulation models real gases, where the assumptions of the ideal gas law ($PV=nRT$) that the gas molecules have no volume and no intermolecular forces breaks down. 

The compressibility factor ($Z$), also known as the gas deviation factor, is a dimensionless quantity used to measure how much a real gas deviates from ideal gas behaviour.

$$ Z = \frac{PV}{nRT} = \frac{P_{real}}{P_{ideal}} $$

*   **If $Z=1$:** the gas behaves ideally.
*   **If $Z\neq 1$:** the gas is non-ideal. The value of Z depends on the identity of the gas, the pressure, and temperature.

The scale of Z tells us which forces are dominating the gas behaviour.

A. $Z<1$: 
*   The dominant forces are the attractive forces between molecules - the molecules attract each other, "pulling" the gas together and making the actual volume smaller than the predicted ideal volume.
*   This usually occurs at moderate pressures and low temperatures where molecules are close enough and moving slowly enough for attractions to matter.

B. $Z>1$:
*   The dominant forces are the repulsive forces and molecular volume (excluded volume).
*   At high pressures, molecules are packed so tightly that their own physical size becomes significant. They resist being compressed further, making actual volume larger than ideal volume.
*   This occurs at very high pressures or for gases like $H_2$ and $He$ even at lower pressures, because their attractions are extremely weak.

**Graphical Behaviour**

If you plot $Z$ on the y-axis and pressure on the x-axis:
1. Low pressure ($P\rightarrow0$): all gases converge to $Z=1$ because the molecules are so far apart that they behave ideally.
2. Moderate pressure: most gases ($O_2$, $N_2$, $CH_4$, etc.) show a 'dip' where $Z$ drops below 1 due to attractions.
3. High pressure: the curve eventually crosses $Z=1$ and shoots upward ($Z>1$) as the physical volume of the molecules prevents further compression.

As temperature increases, the "dip" becomes shallower. At very high temperatures, the gas behaves more ideally over a wider range of pressures.
*   This is because at very high temperatures, gas molecules have much greater kinetic energy, which reduces the influence of intermolecular forces that cause deviation from ideal behaviour.

The Boyle Temperature ($T_B$) is the specific temperature at which a real gas behaves most like an ideal gas over a wide range of pressures.
$$T_B=\frac{a}{Rb}$$
Where $a$ and $b$ are the Van der Waals constants.

The function `` was implemented. This function plots $Z$ against $P$

### 4.7 Negative Activation Energy
The oxidation of nitric oxide is famous for a counter-intuitive property: **the reaction slows down as temperature increases.** This seems to violate the Arrhenius law, where rate constants ($k$) increase with $T$.

**The Explanation**

The overall rate law derived from the SSA is:
$$ Rate \approx \left( \frac{k_1 k_2}{k_{-1}} \right) [NO]^2 [O_2] $$

The "effective" rate constant is $k_{eff} = \frac{k_1 k_2}{k_{-1}} = K_{eq,1} \times k_2$.
*   **Step 1 ($2NO \rightleftharpoons N_2O_2$):** this equilibrium is exothermic ($\Delta H < 0$). By Le Chatelier's Principle, increasing $T$ shifts the equilibrium to the *left*, dramatically reducing the concentration of the intermediate $N_2O_2$.
*   **Step 2 ($N_2O_2 + O_2 \to 2NO_2$):** this step behaves normally; $k_2$ increases with $T$.

**The Conflict:** The drop in $[N_2O_2]$ (thermodynamics) outweighs the increase in $k_2$ (kinetics). The "fuel" for the second step dries up faster than the engine speeds up.

### 4.8 Advanced Visualisation & Control
The simulation includes several features for controlling the simulation parameters and tailoring the visual output.

1. **Maximum for Concentration Sliders:**
* The maximum value for the concentration sliders is not fixed. A dedicated "Conc. Slider Max" control allows the user to dynamically adjust the input range, enabling the setup of both dilute and highly concentrated systems without being constrained by arbitrary limits.

2. **Y-Axis Scaling:**
* When modelling systems with species whose concentrations are orders of magnitude different (e.g., catalysts or bulk reactants), standard plots can be misleading. The "Exclude from Y-Axis Scaling" checkboxes allow users to instruct the plotting engine to ignore specific species when calculating the y-axis limits for the concentration plot. The excluded species are still plotted, but they do not dominate the scale.

3. **User-Defined End Time:**
The "End Time (s)" input allows for direct control of the simulation's duration. This allows for two distinct modes of operation:
*   **Run-to-Completion (default):** if left empty, the simulation runs until the system reaches equilibrium or a steady state.
*   **Fixed-Time Experiment:** if a time is specified, the simulation runs for exactly that duration. This is essential for studying non-convergent systems like chemical oscillators or for analysing the initial kinetics of a slow reaction.

4. **Intelligent Rate Plot Scaling:**
* Reaction rates can spike to extremely high values during the initial transient phase (the first few microseconds or nanoseconds) before settling. To prevent this initial spike from compressing the y-axis of the rate plot, the visualisation engine automatically identifies and excludes these extreme initial outliers from the primary y-axis calculation, providing a clear view of the reaction rates during the bulk of the simulation, while still allowing the user to zoom in and view the initial spike if desired.

5. **Plot Visibility Controls:**
Specific chemical species can be completely hidden from the plot, which can be useful for focusing on key reactants and products while hiding transient intermediates or constant pool chemicals. The species' line will disappear from the concentration graph, and its entry will be removed from the legend.

## 5. Key Architectural Changes (from the previous model)
1.  **Object-oriented chemistry:** molecules and reactions are now objects with their own properties, rather than just variables in a function.
*   This allows any combination of elementary steps to be modelled.
2.  **The summation principle:** the rate of change for any species is calculated dynamically by summing the contributions of every reaction it participates in:
    $$ \frac{dn_i}{dt} = V \sum_{j} \nu_{ij} r_j $$
*   This allows the engine to automatically handle coupled mechanisms and other complex reactions without explicit human instruction.
3.  **Implicit solver:** to handle the vast difference in timescales between fast equilibria ($10^{-6}s$) and slow oxidations ($10s$), we switch from explicit solvers to the **Radau IIA** method (an implicit Runge-Kutta method) which is designed specifically for stiff systems.

The simulation engine has been refactored to adhere to best practices ensuring it is not only scientifically accurate but also maintainable, scalable, and efficient.

### 1. The DRY Principle (Don't Repeat Yourself)
*   **Single source of truth:** physical constants (like $R$) and equilibrium pairs are defined once and referenced everywhere.
*   **Centralised logic:** for example, thermodynamic calculations ($Q_c, K_p$) are handled by a single helper method (`_calculate_mass_action_ratio`) rather than copied multiple times. This prevents bugs where one equation is updated but another is forgotten.

### 2. Separation of Concerns
*   The `ChemicalSystem` class handles the physics, the `generate_simulation_plot` function handles the visualisation, while the `create_interface` handles the user input, etc.

### 3. Vectorisation (NumPy)
*   Instead of looping through Python lists (which is slow), NumPy arrays are used for bulk calculations. This leverages C-level optimization, allowing the solver to handle stiff systems with thousands of time steps in milliseconds.

### 4. Stability/Safety
*   Results are automatically cleaned of duplicate time points to prevent divide by zero errors.
*   **Defensive programming:** functions like `calculate_thermodynamics` include safety clamps (`1e-20`) to ensure logarithmic plots do not crash on zero concentrations.

Furthermore, before any math begins, the `_validate_reactions` method performs an integrity check where it iterates through every reactant and product in the reaction list and ensures they exist in the `species_list`. This prevents ghost species from corrupting the stoichiometry matrix.

In [3]:
from chemical_engine import ChemicalSpecies, Reaction, find_all_equilibrium_pairs, ChemicalSystem, PerturbationSimulation
from visualisation import sanitise_results, find_completion_time, align_previous_run, generate_table, generate_plot, find_key_events, create_interface
from validation_toolkit import diagnose_and_verify_steady_state, diagnose_real_gas_divergence

## Execution Block
### User Guide
To model a new chemical system, follow this 4-step recipe. This replaces the hard-coded variables of the previous models.

1.  **Define Species:** instantiate `ChemicalSpecies` objects.
    *   You must provide Van der Waals constants (`vdw_a`, `vdw_b`). If unknown, use `0.0` (treat as Ideal Gas), though this disables real pressure accuracy.
2.  **Define Reactions:** instantiate `Reaction` objects.
    *   **Stoichiometry:** use dictionaries, e.g., `{'NO': 2}` means $2NO$.
    *   **Kinetics:** provide $A$ and $E_a$ (activation energy in J/mol).
    *   *Note:* reversible reactions must be defined as two separate objects - one forward, one reverse.
3.  **Define initial state:** create a dictionary mapping species names to initial moles.
4.  **Instantiate system:** pass these lists into `ChemicalSystem`.

#### Important Note on Units
The engine solves for concentration in molarity. Therefore, users must ensure that the pre-exponential factor ($A$) is provided in consistent units:
*   For 1st order reactions: $s^{-1}$.
*   For 2nd order reactions: $dm^3 mol^{-1} s^{-1}$.
*   For 3rd order reactions: $dm^6 mol^{-2} s^{-1}$.
The engine does not currently perform automatic unit conversion.

In [4]:
N2 = ChemicalSpecies('N2', vdw_a=1.352, vdw_b=0.0387)
H2 = ChemicalSpecies('H2', vdw_a=0.244, vdw_b=0.0266)
NH3 = ChemicalSpecies('NH3', vdw_a=4.170, vdw_b=0.0371)

R = 8.314
T_ref = 900

Ea_f = 85000  # J/mol
A_f = 0.1 * np.exp(Ea_f / (R * T_ref))

Ea_r = 177000 # J/mol
A_r = 2.5 * np.exp(Ea_r / (R * T_ref))

r_fwd = Reaction(
    reactants={'N2': 1, 'H2': 3}, 
    products={'NH3': 2}, 
    A=A_f, 
    Ea=Ea_f
)

r_rev = Reaction(
    reactants={'NH3': 2}, 
    products={'N2': 1, 'H2': 3}, 
    A=A_r, 
    Ea=Ea_r
)

initial_moles = {'N2': 1.0, 'H2': 3.0, 'NH3': 0.0}

haber_system = ChemicalSystem(
    species_list=[N2, H2, NH3],
    reaction_list=[r_fwd, r_rev],
    initial_moles=initial_moles,
    initial_V=1.0,
    initial_T=700 # typical industrial temperature
)

create_interface(haber_system)
diagnose_and_verify_steady_state(haber_system)

Verifying the steady state...

DIAGNOSIS: No coupled equilibrium/drain mechanisms were found.
The system is either a simple equilibrium or a series of irreversible steps.


## Verification and Testing
The `SystemTester` class is a validation framework designed to ensure that the `ChemicalSystem` engine adheres to fundamental physical laws. These tests check for scientific violations (e.g, creation of matter, thermodynamic inconsistencies, or numerical instability).

### Adaptive Simulation Strategy
One of the key innovations of this class is how it automatically detects complex mechanisms.
*   **The problem:** standard chemical systems eventually reach equilibrium, but complex systems (such as oscillating reactions) may continue oscillating indefinitely. A "convergence check" causes oscillators to run until they hit the iteration limit or crash.
*   **The solution:** upon initialisation, the tester scans the reaction network (and stoichiometry matrix) for autocatalysis - specifically, reactions where a species appears as both a reactant and a product with a higher coefficient ($X\rightarrow2X$).
    * Standard systems: configured to run in the normal convergence mode (simulates until $dn/dt<\text{tolerance}$).
    * Complex systems: configured to run in fixed-time mode to capture the oscillation without forcing early termination.

### Test Suite Breakdown
1. **Mass Conservation:**
*   The Law of Conservation of Mass states that matter cannot be created or destroyed in an isolated chemical system.
*   This test calculates the total mass of the system ($\Sigma n_i\times MW_i$) at the start ($t=0$) and the end ($t_{final}$) of the simulation.
*   The difference between initial and final mass must be less than a certain tolerance. Failure indicates a stoichiometry error in the reaction definitions (e.g., an unbalanced equation) or significant numerical drift in the integrator.
*   The `test_mass_conservation` function does not simply sum the moles (which changes during reactions such as $A\rightarrow2B$). The summation acts as a checksum for the solver.

2. **Kinetic Convergence:**
*   This `test_kinetic_convergence` function calculates the instantaneous derivative $dn/dt$ for every species using the final state vector. The maximum absolute rate of change must be less than a certain tolerance for standard systems.
*   Failure indicates the simulation terminated prematurely (likely due to the `max_iterations` limit) before the chemistry actually finished.
*   If autocatalysis is found, the tester expects instability. In that case, rather than checking for zero rates, the test calculates the gradient of species concentrations and counts sign flips (peaks/troughs). It passes if the system exhibits stable oscillation ($>2$ phase changes).
    * Fail condition: the system explodes (values$\rightarrow\infty$) or decays immediately to zero.

3. **Thermodynamic Equilibrium & Steady State:**
This `test_thermodynamic_equilibrium` function verifies that the final concentrations satisfy the Law of Mass Action. It distinguishes between two scenarios:
*   Scenario A: True Equilibrium (system is reversible and closed)
    * Test checks that $Q_c$ equals $K_c$ at the end.
*   Scenario B: Pseudo-Steady State

4. **Real Gas Physics:**
*   This `test_real_gas_physics` function forces a simulation into a highly compressed state ($V\approx0.05 L$) and calculates ideal pressure and real pressure.
*   Validates whether the engine is correctly considering intermolecular forces and molecular volume.
*   Automatically skips execution if species are defined as ideal gases (Van der Waals constants $a=0$, $b=0$).

5. **Le Chatelier's Principle (static method):**
*   A simple $A\rightleftharpoons2B$ equilibrium is simulated (as more complex mechanisms, such as those involving steady states, can exhibit counter-intuitive behaviour that violates the simple heuristic of Le Chatelier's principle).
*   A volume perturbation is applied, and the test checks whether the system responds correctly.

6. **Stiffness Test (static method):**
*   This test runs the following reaction:
$$\text{fast}\xrightarrow{k=10^{12}}\text{intermediate}\xrightarrow{k=0.1}\text{slow}$$
*   The pass condition is that the solver does not crash, the "fast" species is completely consumed, and that the "slow" product has begun to form.
*   Failure indicates that the numerical solver is incorrect or the timestep adaptation is unable to handle stiffness.


In [5]:
from validation_toolkit import SystemTester

no_oxidation_masses = {'NO': 30.01, 'O2': 32.00, 'N2O2': 60.02, 'NO2': 46.01}
# tester = SystemTester(no_oxidation_system, no_oxidation_masses)
# tester.run_all_tests()
SystemTester.run_stiffness_test()

(True, 'PASS: Solver handled stiffness properly.')

## 6. Model Assumptions and Limitations
While this model represents a significant leap in complexity from the previous single-reaction model, it still operates under specific physical assumptions.

### 6.1 Thermodynamics
*   **Isothermal operation:** the simulation treats the reactor as perfectly isothermal ($dT/dt = 0$ unless externally perturbed). It does not account for the **enthalpy of reaction** ($\Delta_r H$). In reality, the exothermic formation of $NO_2$ would release heat, raising the temperature and altering the rate constants dynamically.
*   **Constant enthalpy:** we assume $\Delta H$ is constant over the temperature range. Strictly, $\Delta H$ varies with temperature according to Kirchhoff's Law.
*   **Gas behaviour:** although Van der Waals corrections are added, these are basic and do not currently account for fugacity or non-ideal mixing in high-pressure environments.

### 6.2 Phase Behaviour
*   **Static phase assignment:** species are initialised with a fixed phase (gas, liquid, solid, aqueous, solvent). The model does not simulate dynamic phase transitions (e.g., boiling, melting, condensation) driven by temperature changes unless explicitly modelled as a chemical reaction (e.g., $A(g)\rightleftharpoons A(l)$).
*   **Latent heat:** enthalpy of fusion/vaporisation is currently ignored.

### 6.3 Kinetics
*   **Mass action law:** we assume all reactions follow elementary rate laws ($Rate = k[A]^a[B]^b$). While valid for the elementary steps we defined, real heterogeneous catalysis (like on a converter surface) could follow more complex kinetics which this engine does not currently model.
    * At low pressures, the collision frequency could limit the rate, causing the reaction to shift toward second-order behaviour (the "fall-off regime").
    * Approximations or other complex rate laws are not supported unless written as elementary steps.
*   **Perfect mixing:** the system is modelled as a batch reactor with perfect mixing. It ignores spatial effects, diffusion limitations, or concentration gradients that would exist in a real reactors.
*   **Temperature dependence of the pre-exponential factor:** this simulation assumes that $A$ is temperature-independent. However, this is not true and the assumption only holds over small temperature ranges.

#### Activity vs. Concentration
The engine calculates reaction rates using concentrations. In thermodynamics, the driving force for a reaction is the chemical potential, which depends on activity ($a=\gamma[C]$), not just concentration. In highly non-ideal regimes (e.g., high-pressure gases where $Z\neq1$), the activity coefficient $\gamma$ deviates from 1.0. 

## The Lotka-Volterra System
To test the model further, a simplified model for an oscillating chemical reaction, an abstraction of the famous Lotka-Volterra predator-prey equations, was created.

### The Chemistry of Oscillating Reactions
The Lotka-Volterra equations were originally developed to model predator-prey population dynamics, but they have a direct and profound analogue in chemistry: autocatalysis.
*   An **autocatalytic reaction** is one where a product of the reaction is also a catalyst for that same reaction, creating a positive feedback loop.
*   The most famous chemical example is the **Belousov-Zhabotinsky (BZ)** reaction, which typically involves the oxidation of malonic acid by bromate in an acidic solution, catalysed by cerium or manganese ions. If this reaction is performed in a petri dish, a simple colour change won't be observed; instead, you will see stunning, dynamic, evolving spirals and waves of colour as the concentrations of the species oscillate, creating a chemical clock. 

### Our Implementation
**Species:**
*   `X`: the "prey" molecule.
*   `Y`: the "predator" molecule.
*   `P`: a stable "product".

**Reactions (all irreversible):**
*   `X -> 2X` (prey reproduction, autocatalytic formation of X)
    * `A = 1.0`, `Ea = 0`
    * This is exponential growth - the more `X` there is, the faster more `X` is made.
*   `X + Y -> 2Y` (predator "eats" prey to reproduce)
    * `A = 0.02`, `Ea = 0`
    * Molecule `Y` is also autocatalytic, but it requires `X`as a feedstock. The more `X` (prey) available, the faster `Y` (the predator) can reproduce. In doing so, it consumes `X`.
*   `Y -> P` (predator "dies" and forms a stable product)
    * `A = 1.0`, `Ea = 0`
    * The predator molecule `Y` is unstable and spontaneously decays into an inert product `P`. This is the negative feedback loop that prevents `Y` from consuming all of `X` and growing infinitely.

The oscillation emerges from the interplay of these feedback loops. The population of `X` grows, providing "food" for `Y`. The population of `Y` then grows, consuming `X`. The depletion of `X` starves `Y`, causing its population to crash. With the predator `Y` gone, `X` is free to grow again, and the cycle repeats.

**Initial Conditions:**
*   `initial_moles = {'X': 50.0, 'Y': 20.0, 'P': 0.0}`
*   `initial_V = 1.0`
*   `initial_T = 300`

**Expected behaviour:** this system will *not* react a simple equilibrium. The concentrations of X and Y will oscillate over time. This test was designed to test the robustness of the test suite. 

In [6]:
X = ChemicalSpecies('X', vdw_a=0, vdw_b=0)
Y = ChemicalSpecies('Y', vdw_a=0, vdw_b=0) 
P = ChemicalSpecies('P', vdw_a=0, vdw_b=0)

prey_reproduction = Reaction({'X': 1}, {'X': 2}, A=1.0, Ea=0)
prey_consumption = Reaction({'X': 1, 'Y': 1}, {'Y': 2}, A=0.02, Ea=0)
predator_death = Reaction({'Y': 1}, {'P': 1}, A=1.0, Ea=0)

initial_moles = {'X': 50.0, 'Y': 20.0, 'P': 0.0}

lotka_volterra_system = ChemicalSystem(
    species_list=[X, Y, P],
    reaction_list=[prey_reproduction, prey_consumption, predator_death],
    initial_moles=initial_moles,
    initial_V=1.0, 
    initial_T=300)

lotka_volterra_masses = {'X': 1.0, 'Y': 1.0, 'P': 1.0}
tester = SystemTester(lotka_volterra_system, lotka_volterra_masses)
tester.run_all_tests()


                      SYSTEM DIAGNOSTICS REPORT                       
Starting fixed-time simulation at T=300K for 500.0s...
Mass conservation...           | FAIL     | FAIL: Mass conservation violation. Delta: 2.51e+04 g.
Starting fixed-time simulation at T=300K for 500.0s...
Kinetic convergence...         | PASS     | PASS: System exhibits stable oscillation (151 phase changes detected).
Thermodynamic equilibrium...   | PASS     | SKIPPED: No reversible reactions defined in system.
Real gas physics...            | PASS     | SKIPPED: System uses ideal gas assumptions (all VdW constants are 0).
DEBUG: Calculating rates using external results (time points: 102)
Le Chatelier's principle...    | PASS     | PASS: Synthetic A<->2B system shifted right (dn=+1.10e-01) on expansion.
Stiffness stability...         | PASS     | PASS: Solver handled stiffness properly.
----------------------------------------------------------------------
OVERALL RESULT: 5/6 tests passed.



**Interpretation of Results:**

The Lotka-Volterra system will fail the mass conservation test because it models an open biological system, not a closed chemical system. 

*Why?*

The mass conservation test is predicated on the fundamental axiom of a closed thermodynamic system: matter is neither created nor destroyed. The total mass at time t must equal the total mass at t=0.

However, the Lotka-Volterra equations include a step that explicitly violates this law in the prey reproduction step:
$$X\rightarrow2X$$

This reaction seems to be explicitly creating matter out of nothing.
*   **Biologically**, this makes sense. Rabbits eat grass to reproduce. The "mass" comes from the grass. However, in the Lotka-Volterra model, the grass is implicit (it is assumed to be infinite and is not modeled as a chemical species). The equation is a mathematical abstraction of a biological or chemical system where `X` is formed from a precursor, `A`, which is assumed to be in such vast excess that its concentration is constant. This is known as the "pool chemical" or "chemostat" approximation. The true reaction is `A + X -> 2X`, but since `[A]` is assumed constant, it is absorbed into the rate constant.
*   **Chemically**, because the "grass" reactant is missing from the equation, the engine sees mass appearing from a vacuum. The `SystemTester` sums up the total mass of the system. Since $n_X$ increases during reproduction without a corresponding decrease in another species, the total mass fluctuates, causing the test to fail.

This failure is actually a success for the `SystemTester` class. It correctly identified that the reaction being modelled is not a valid, closed thermodynamic system.

We can model a version that explicitly accounts for the "grass". To address the mass conservation failure, the system can be refactored into a closed, four-species cycle. The new system comprises a "nutrient" or "grass" species (`G`), which is consumed by the "prey" (`X`) to reproduce. The "predator" (`Y`) consumes X to reproduce, and then "dies" to form an inert "product" (`P`). The closing step is the decomposition of `P` back into the nutrient `G`, ensuring every atom is accounted for throughout the simulation.

**Reaction Scheme:**

*   `G + X -> 2X` (prey consumes nutrient to reproduce)
*   `X + Y -> 2Y`
*   `Y -> P`
*   `P -> G` (product decomposes, regenerating nutrient)

The mass conservation test will now be passed.

In [7]:
X = ChemicalSpecies('X', vdw_a=0, vdw_b=0)
G = ChemicalSpecies('G', vdw_a=0, vdw_b=0, species_type='pool') # nutrient/grass
Y = ChemicalSpecies('Y', vdw_a=0, vdw_b=0) 
P = ChemicalSpecies('P', vdw_a=0, vdw_b=0)

r1 = Reaction({'G': 1, 'X': 1}, {'X': 2}, A=1.0, Ea=0)
r2 = Reaction({'X': 1, 'Y': 1}, {'Y': 2}, A=1.0, Ea=0)
r3 = Reaction({'Y': 1}, {'P': 1}, A=1.0, Ea=0)
r4 = Reaction({'P': 1}, {'G': 1}, A=0.5, Ea=0) # decomposition step

initial_moles = {'X': 50.0, 'G': 200.0, 'Y': 20.0, 'P': 0.0}

closed_lotka_volterra_system = ChemicalSystem(
    species_list=[X, G, Y, P],
    reaction_list=[r1, r2, r3, r4],
    initial_moles=initial_moles,
    initial_V=1.0, 
    initial_T=300)

lotka_volterra_masses = {'X': 1.0, 'G': 1.0, 'Y': 1.0, 'P': 1.0}
tester = SystemTester(closed_lotka_volterra_system, lotka_volterra_masses)
tester.run_all_tests()
create_interface(closed_lotka_volterra_system)


                      SYSTEM DIAGNOSTICS REPORT                       
Starting fixed-time simulation at T=300K for 500.0s...
Mass conservation...           | FAIL     | FAIL: Mass conservation violation. Delta: 7.00e+01 g.
Starting fixed-time simulation at T=300K for 500.0s...
Kinetic convergence...         | PASS     | PASS: Autocatalytic system settled to steady state.
Thermodynamic equilibrium...   | PASS     | SKIPPED: No reversible reactions defined in system.
Real gas physics...            | PASS     | SKIPPED: System uses ideal gas assumptions (all VdW constants are 0).
DEBUG: Calculating rates using external results (time points: 102)
Le Chatelier's principle...    | PASS     | PASS: Synthetic A<->2B system shifted right (dn=+1.10e-01) on expansion.
Stiffness stability...         | PASS     | PASS: Solver handled stiffness properly.
----------------------------------------------------------------------
OVERALL RESULT: 5/6 tests passed.



### The Brusselator
This is a theoretical model which produces oscillations. It is defined by four elementary steps. In the simulation, `A` and `B` are pool chemicals whose concentrations are constant throughout the whole simulation (simulating an open system with a continuous supply of reactants). `X` and `Y` are the oscillating intermediates, and `P` is a final, inert product.

#### The Mechanism:
* **(R1) Initiation:** `A -> X`
    * This step continuously feeds intermediate `X` into the system at a constant rate, fueled by pool chemical `A`.
* **(R2) "Tri-Molecular Step":** `2X + Y -> 3X`
    * This step is autocatalytic, where `X` catalyses its own production. The rate is highly non-linear ($rate\propto[X]^2[Y]$), causing an explosive increase in $[X]$ when conditions are right.
* **(R3) Feedback/Inhibition:** `B + X -> Y + D`
    * This step consumes `X` to produce the other intermediate, `Y`. It acts as a feedback loop: the production of `Y` is necessary for the autocatalytic step, but this step also remove `X`.
* **(R4) Removal:** `X -> P`
    * This step is a simple, unimolecular decay that removes `X` from the system, preventing its concentration from growing indefinitely.

Under conditions where A and B are in vast excess (and can therefore be modeled at constant concentration), the rate equations are:
* $\frac{d[X]}{dt}=[A]+[X]^2[Y]-[B][X]-[X]$
* $\frac{d[Y]}{dt}=[B][X]-[X]^2[Y]$
where the rate constants have been set to 1 for convenience.

The Brusselator has a fixed point (where the state of the system does not change with time) at $[X]=[A]$ and $[Y]=[B]/[A]$. This fixed point becomes unstable when $[B]>1+[A]^2$ leading to an oscillation of the system. Unlike the Lotka-Volterra system, the oscillations of the Brusselator do not depend on the amount of reactant present initially. Instead, after sufficient time, the oscillations approach a limit cycle, in which the system oscillates periodically with fixed amplitude and period. Once reached, the system keeps cycling forever.

This can be seen in a graph from the simulation, which would show the stable, repeating cycle (limit cycle). The concentrations of `A` and `B` will remain perfectly constant. The total pressure (dotted teal line) rises over the course of the simulation. This is because the net reactions produce the final product P, continuously increasing the total number of moles of gas in the fixed volume. The smaller oscillations on the pressure trace correspond to the fluctuations in the total moles of the intermediates X and Y.

In [8]:
A = ChemicalSpecies('A', species_type='pool', phase='aqueous')
B = ChemicalSpecies('B', species_type='pool', phase='aqueous')
X = ChemicalSpecies('X', phase='aqueous')
Y = ChemicalSpecies('Y', phase='aqueous')
P = ChemicalSpecies('P', phase='aqueous') # P is a product sink

A_param = 1.0
B_param = 3.0
scaling_factor = 0.1 # slows down the reaction to make it easier to solve

r1 = Reaction({'A': 1}, {'X': 1}, A=1.0 * scaling_factor, Ea=0)
r2 = Reaction({'X': 2, 'Y': 1}, {'X': 3}, A=1.0 * scaling_factor, Ea=0)
r3 = Reaction({'B': 1, 'X': 1}, {'Y': 1, 'P': 1}, A=1.0 * scaling_factor, Ea=0)
r4 = Reaction({'X': 1}, {'P': 1}, A=1.0 * scaling_factor, Ea=0)

initial_moles_brusselator = {
    'A': A_param,
    'B': B_param,
    'X': 1.5,
    'Y': 3.0,
    'P': 0.0
}

brusselator_system = ChemicalSystem(
    species_list=[A, B, X, Y, P],
    reaction_list=[r1, r2, r3, r4],
    initial_moles=initial_moles_brusselator,
    initial_V=1.0,
    initial_T=300,
    method='BDF',     
    rtol=1e-4,     
    atol=1e-7
)

create_interface(brusselator_system)

### Belousov-Zhabotinsky (BZ) Reaction
This is a prime example of an oscillating reaction, a system that does not proceed to a simple, static equilibrium but instead exhibits periodic fluctuations in the concentrations of its intermediates. These oscillations are visible as dynamic colour changes or propagating waves.

Modelling the full BZ mechanism is computationally intensive and requires knowledge of numerous, often poorly constrained, rate constants. The **Oregonator model** is a remarkable simplification that captures the essential non-linear feedback loops of the BZ reaction with just five core elementary steps and a handful of adjustable parameters. It is a reduced model of the FKN mechanism, which still involved eleven reactions and eighteen elementary steps.

#### The Species:
The model involves the following key species, which are abstractions of the real chemical players in the bromate-malonic acid system:
* `X`: the activator species (analogous to $HBrO_2$).
* `Y`: the inhibitor species (analogous to $Br^-$).
* `Z`: the oxidised form of the catalyst (analogous to $Ce^{4+}$).
* `A`: the primary reactant, treated as a "pool chemical" (analogous to $BrO_3^-$). Its concentration is assumed to be constant.
* `P`: an inert product.

#### The Mechanism:
1. `A + Y -> X + P`: the inhibitor `Y` is consumed to produce the activator `X` and the inert product `P`.
2. `X + Y -> 2P`: the activator and inhibitor annihilate each other.
3. `A + X -> 2X + 2Z`: this is autocatalysis (positive feedback). The activator `X` reacts with the primary reactant `A` to produce *more* `X`, leading to an exponential increase in its concentration. This step also produces the oxidised catalyst `Z`.
4. `2X -> A + P`: the activator is consumed (self-limiting step).
5. `Z -> fY`: this is inhibition (negative feedback). The oxidised catalyst `Z` decomposes to regenerate the inhibitor `Y`. The stoichiometric factor `f` is an adjustable parameter that controls the strength of this feedback.

The combination of these steps creates the oscillation. A small amount of `X` is produced (step 1). This triggers the explosive autocatalytic production of more `X` (step 3), causing `[X]` and `[Z]` to spike. The high concentration of `Z` then regenerates the inhibitor `Y` (step 5). The now-high concentration of `Y` quenches the production of `X` and consumes it (steps 1 & 2), causing `[X]` to crash. With `[X]` gone, `[Y]` is slowly consumed, and the cycle begins anew.

#### Rate Equations:
* $\frac{d[X]}{dt}=k_1[A][Y]-k_2[X][Y]+k_3[A][X]-2k_4[X]^2$
* $\frac{d[Y]}{dt}=-k_1[A][Y]-k_2[X][Y]+fk_5[Z]$
* $\frac{d[Z]}{dt}=2k_3[A][X]-k_5[Z]$

#### Notes:
* This system is stiff, with periods of slow change followed by explosive growth and rapid crashes. An implicit solver is needed.
* The system will not reach a simple equilibrium. Therefore, the simulation must be run in fixed-time mode (`t_end`) to capture the oscillatory behaviour without running indefinitely.
* The concentrations of the intermediates (`X`, `Y`, `Z`) fluctuate by many orders of magnitude. A logarithmic y-axis is required for effective visualisation.

In [9]:
A = ChemicalSpecies('A', vdw_a=0, vdw_b=0, species_type='pool')
X = ChemicalSpecies('X', vdw_a=0, vdw_b=0)     # activator
Y = ChemicalSpecies('Y', vdw_a=0, vdw_b=0)     # inhibitor
Z = ChemicalSpecies('Z', vdw_a=0, vdw_b=0)     # oxidised catalyst
P = ChemicalSpecies('P', vdw_a=0, vdw_b=0)     # inert product

# units: M^-1 s^-1 for k1 to k4, s^-1 for k5
k1 = 0.08
k2 = 1.6e9 
k3 = 480
k4 = 4e7
k5 = 1.0 

f = 1.0 # stoichiometric factor

r1 = Reaction(reactants={'A': 1, 'Y': 1}, products={'X': 1, 'P': 1}, A=k1, Ea=0)
r2 = Reaction(reactants={'X': 1, 'Y': 1}, products={'P': 2}, A=k2, Ea=0)
r3 = Reaction(reactants={'A': 1, 'X': 1}, products={'X': 2, 'Z': 2}, A=k3, Ea=0)
r4 = Reaction(reactants={'X': 2}, products={'P': 1}, A=k4, Ea=0)
r5 = Reaction(reactants={'Z': 1}, products={'Y': f}, A=k5, Ea=0)

initial_moles = {'A': 3.0, 'X': 0.01, 'Y': 0.01, 'Z': 0.01, 'P': 0.0}

oregonator_system = ChemicalSystem(
    species_list=[A, X, Y, Z, P],
    reaction_list=[r1, r2, r3, r4, r5],
    initial_moles=initial_moles,
    initial_V=1.0,
    initial_T=300,
)

create_interface(oregonator_system)

## Catalysed Conversion of $A\rightarrow P$
**Overall Reaction:** $A\xrightarrow{\text{cat}}P$

### Mechanism
1. Formation of an intermediate.
* $A+C\rightleftharpoons AC$, where $C$ is the catalyst and $AC$ is an intermediate.
2. Product formation and catalyst regeneration.
* $AC\rightarrow P+C$

### Rate Equations
Let rate constants be $k_1$, $k_{-1}$, $k_2$:
* $\frac{d[A]}{dt}=-k_1[A][C]+k_{-1}[AC]$
* $\frac{d[AC]}{dt}=k_1[A][C]-(k_{-1}+k_2)[AC]$
* $\frac{d[P]}{dt}=k_2[AC]$

In [10]:
A = ChemicalSpecies('A', vdw_a=0, vdw_b=0)
C = ChemicalSpecies('C', vdw_a=0, vdw_b=0)
AC = ChemicalSpecies('AC', vdw_a=0, vdw_b=0)
P = ChemicalSpecies('P', vdw_a=0, vdw_b=0)

r1_fwd = Reaction({'A': 1, 'C': 1}, {'AC': 1}, A=1.0e6, Ea=25000)
r1_rev = Reaction({'AC': 1}, {'A': 1, 'C': 1}, A=5.0e4, Ea=35000)
r2 = Reaction({'AC': 1}, {'P': 1, 'C': 1}, A=1.0e5, Ea=30000)

initial_moles = {'A': 1.0, 'C': 1.0, 'AC': 0.0, 'P': 0.0}

test_system = ChemicalSystem(
    species_list=[A, C, AC, P],
    reaction_list=[r1_fwd, r1_rev, r2],
    initial_moles=initial_moles,
    initial_V=1.0,
    initial_T=298)

create_interface(test_system)

## The Contact Process
The Contact Process is used to produce sulfuric acid. Sulfuric acid is one of the most important industrial chemicals, so much so that it is sometimes used as a rough indicator of a country's industrial strength as it is a fundamental, widely used chemical in nearly all industrial sectors. It is essential to food production, used in fertilisers, mining, manufacturing (from dyes, pigments, detergents, explosives, pharmaceuticals, to synthetic fibres), batteries, waste treatment, petroleum refining, and more. Chemically, it is a strong acid and a strong dehydrating agent, and important for many reactions.

The Contact Process is used as it is extremely efficient, with high conversion, low waste, and high purity acid produced. It also requires only cheap raw materials and is thermodynamically favourable and scalable.

## Iodine Clock Reaction
This is a famous chemical demonstration where two colourless solutions are mixed, and after a predictable delay, the solution suddenly flashes to a deep blue colour (due to the formation of a triodide-starch complex).

There are many different variations of this, such as the hydrogen peroxide variation (involving a solution of hydrogen peroxide and sulfuric acid), iodate variation, chlorate variation, and more. 

### The Mechanism (Persulfate)
This clock reaction uses sodium, potassium, or ammonium persulfate.

The core principle is a competition between two reactions: a slow one that *produces* iodine (this is the RDS) and an extremely fast one that immediately *consumes* it.
1. Persulfate ions slowly oxidise iodide ions to produce iodine.
    * $S_2O_8^{2-}+2I^-\xrightarrow{k_{slow}}2SO_4^{2-}+I_2$
2. As soon as any $I_2$ is formed, it is immediately and rapidly reduced back to iodide by thiosulfate ions.
    * $I_2+2S_2O_3^{2-}\xrightarrow{k_{fast}}2I^-+S_4O_6^{2-}$

The solution remains colourless because the concentration of $I_2$ is kept near zero by the thiosulfate. The "clock" goes off the instant the thiosulfate is completely consumed. At that moment, the fast reaction stops and the $I_2$ produced by the slow reaction suddenly begins to accumulate, turning the reaction blue.

## Free Radical Substitution

## Catalytic Ozone Depletion
The destruction of ozone is a free radical mechanism. A single chlorine radical (`Cl·`) acts as the catalyst. These radicals are created when chlorofluorocarbons (CFCs) undergo photolysis in the atmosphere. CFCs were used in aerosols, as coolants in fridges, and as solvents in industry before they were banned after evidence they damaged the ozone layer.

The overall reaction is: $2O_3\rightarrow3O_2$. The depletion of ozone by chlorine is a cycle that is superimposed upon the natural ozone life cycle, known as the Chapman cycle. In the stratosphere, there is a dynamic equilibrium where UV light creates atomic oxygen (`O·`) and ozone (`O₃`), and they interconvert. The key insight from Molina and Rowland, for which they won the Nobel Prize, is that chlorine radicals can catalytically destroy ozone by short-circuiting this natural cycle.

### Simplified Chapman Cycle
n the stratosphere, there is a natural, dynamic balance between ozone (`O₃`), molecular oxygen (`O₂`), and highly reactive atomic oxygen (`O·`).
* **R1 (Forward/Reverse):** $O_3\xrightleftharpoons[k_{-1}]{k_1}O_2+O·$

### Chlorine Catalytic Cycle
* **R2:** $Cl·+O_3\xrightarrow{k_2}ClO·+O_2$ (chlorine monoxide is formed)
* **R3:** $ClO·+O·\xrightarrow{k_3}Cl·+O_2$ (chlorine monoxide reacts with atomic oxygen, regenerating the original chlorine radical)
The net reaction is: $O_3+O·\rightarrow2O_2$. The chlorine cycle dramatically accelerates the rate of the naturally occurring ozone destruction reaction.

### Termination
The cycle does not continue forever. Eventually, the reactive radicals combine to form more stable molecules, terminating the chain reaction.
* **R4:** $Cl·+ClO·\xrightarrow{k_4}Cl_2O_2$ (simplified termination step)

### Initial Conditions
Reaction is started at a constant temperature representative of the stratosphere (220 K). At the low pressures of the stratosphere, all species are assumed to behave as ideal gases (Van der Waals constants are set to 0).

In this reaction, `Cl·`, the chlorine radical, is the catalyst and `ClO·`, the chlorine monoxide radical, is an intermediate. 

## Synthesis of Phosgene ($COCl_2$)
This is a reversible gas-phase synthesis from carbon monoxide and chlorine gas:
$$CO(g)+Cl_2(g)\rightleftharpoons COCl_2(g)$$
There is a change in the number of moles of gas ($\Delta n_{gas}=1-(1+1)=-1$). This makes the equilibrium sensitive to changes in pressure and volume.

Injecting a quantity of an inert gas (e.g., argon) at constant volume and temperature would have two effects.
1. The injection of argon at a constant volume 

In [ ]:
s_co    = ChemicalSpecies("CO", vdw_a=1.485, vdw_b=0.0399, phase='gas', molar_mass=28.01)
s_cl2   = ChemicalSpecies("Cl2", vdw_a=6.493, vdw_b=0.0562, phase='gas', molar_mass=70.90)
s_cocl2 = ChemicalSpecies("COCl2", vdw_a=10.2, vdw_b=0.080, phase='gas', molar_mass=98.92)

# Argon is defined as a 'reactant' (so we can inject it and track its pressure), 
# but it will have no reactions defined, so it stays inert.
s_ar    = ChemicalSpecies("Ar", vdw_a=1.345, vdw_b=0.0322, phase='gas', molar_mass=39.95)

species_list = [s_co, s_cl2, s_cocl2, s_ar]

r_fwd = Reaction(
    reactants={'CO': 1, 'Cl2': 1}, 
    products={'COCl2': 1}, 
    A=5.0e4, Ea=15000.0 
)

r_rev = Reaction(
    reactants={'COCl2': 1}, 
    products={'CO': 1, 'Cl2': 1}, 
    A=2.0e10, Ea=105000.0 
)

reactions = [r_fwd, r_rev]

initial_moles = {
    'CO': 1.0,
    'Cl2': 1.0,
    'COCl2': 0.0,
    'Ar': 0.0 # start with empty argon
}

phosgene_sys = ChemicalSystem(
    species_list, reactions, initial_moles, 
    initial_V=1.0, initial_T=600.0
)

create_interface(phosgene_sys)